# RAG Branching Example

A graph that answers questions about this system using RAG on `vector_db/llm_graph_docs`.

**Graph layout:**

```
branch_llm  ──►  check_type (ConditionalNode)
                    │
          ┌─────────┴──────────┐
       coding               general
          │                    │
  coding_retrieval    general_retrieval
          │                    │
    coding_llm          general_llm
```

- **coding** branch: retrieves relevant docs and answers with a code-focused prompt
- **general** branch: retrieves relevant docs and answers with a conceptual explanation prompt

In [1]:
from dotenv import load_dotenv
from IPython.display import Markdown
from openai import OpenAI

from llm_graph.llm.response_functions import OpenAI_response_fn
from llm_graph.factories.llm import create_llm_node
from llm_graph.factories.rag import create_rag_query_pair
from llm_graph.core.nodes import ConditionalNode
from llm_graph.core.graphrunner import GraphRunner
from llm_graph.core.sessionrunner import SessionRunner

In [2]:
load_dotenv()
client = OpenAI()
response_fn = OpenAI_response_fn(client)

## Prompts

In [4]:
branching_prompt = """
You are making a decision on the type of query given.
This will either be a coding query (asking for code examples, implementation details, usage snippets)
or a general query (asking for conceptual explanations, descriptions, or comparisons).

Respond ONLY with valid JSON with key 'query_type' set to either 'coding' or 'general'.

The query is: {user_query}
"""

coding_prompt = """
You are an expert on the llm-graph-engine system.
A user has asked a coding question about this system.
Use the retrieved documentation to give a clear, complete code example that answers the query.
Include imports where relevant. Do not go beyond what is asked.

Respond ONLY with valid JSON with key 'answer' containing your response.

The query is: {user_query}
"""

general_prompt = """
You are an expert on the llm-graph-engine system.
A user has asked a general question about this system.
Use the retrieved documentation to give a clear conceptual explanation.

Respond ONLY with valid JSON with key 'answer' containing your response.

The query is: {user_query}
"""

## Build the graph

In [5]:
# Classifies the query — routes to 'coding' or 'general'
branch_node = create_llm_node(
    response_fn=response_fn,
    name="branch_llm",
    prompt_template=branching_prompt,
    next_node_name="check_type",
)

check_type_node = ConditionalNode(
    name="check_type",
    condition_fn=lambda state: state["query_type"],
)

In [6]:
DB_PATH = "../vector_db"
COLLECTION = "llm_graph_docs"

# Coding branch: retrieval → LLM with coding prompt
coding_nodes = create_rag_query_pair(
    path=DB_PATH,
    collection_name=COLLECTION,
    response_fn=response_fn,
    retrieval_node_name="coding",
    llm_node_name="coding_llm",
    prompt_template=coding_prompt,
)

# General branch: retrieval → LLM with general prompt
general_nodes = create_rag_query_pair(
    path=DB_PATH,
    collection_name=COLLECTION,
    response_fn=response_fn,
    retrieval_node_name="general",
    llm_node_name="general_llm",
    prompt_template=general_prompt,
)

In [7]:
graphrunner = GraphRunner.build(
    node_dicts=[
        {"branch_llm": branch_node, "check_type": check_type_node},
        coding_nodes,
        general_nodes,
    ],
    start_node="branch_llm",
)

session = SessionRunner(
    graph=graphrunner,
    session_keys=["message_history"],
)

## Run some queries

In [8]:
# General question — should route to the 'general' branch
r1 = session.execute({"user_query": "How does GraphRunner.build() work and when would I use it instead of the regular constructor?"})
Markdown(r1["state_dict"]["answer"])

c:\Users\ronsp\micromamba\envs\llm_graph\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6132.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GraphRunner.build() is a convenience constructor for assembling a runnable graph from prebuilt “pieces” (typically dictionaries of nodes) rather than manually wiring everything through the regular GraphRunner constructor.

How it works (conceptually)
- You pass GraphRunner.build() one or more dicts that map node names → node objects.
- It merges those dicts into a single nodes_dict and returns a GraphRunner configured with that combined node set (and whatever start node you specify/it infers).
- This fits the factory pattern used by llm_graph.factories.tool, where “pair” factories (e.g., create_..._pair) return a single dict containing two related nodes intended to be dropped into a larger graph via GraphRunner.build().

When to use build() vs the regular constructor
Use GraphRunner.build() when:
- You are composing a workflow from reusable factory outputs (especially the tool factories’ create_..._pair functions that return dicts “for use with GraphRunner.build()”).
- You want to assemble larger graphs from modular subgraphs (e.g., llm_node + parse_check + retry loop + tool_node + tool_check + analysis_node) without manually merging node maps.

Use the regular constructor when:
- You already have a single complete nodes_dict and want explicit control over initialization.
- You are building graphs “by hand” and don’t need the modular dict-composition style.

Note: Regardless of how you construct it, GraphRunner’s runtime behavior is the same: it executes nodes sequentially, passes the full state_dict into each node, merges partial updates returned by nodes, records a trace_log, tracks in/out tokens per execute() call, and resets node visit counts each execute(). It also does not persist a state_dict across multiple execute() calls.

In [9]:
# Coding question — should route to the 'coding' branch
r2 = session.execute({"user_query": "How do I use create_rag_query_pair to build a RAG workflow?"})
Markdown(r2["state_dict"]["answer"])

```python
from openai import OpenAI

from llm_graph.llm.response_functions import OpenAI_response_fn
from llm_graph.factories.rag import create_rag_query_pair
from llm_graph.core.graphrunner import GraphRunner

# 1) Create an OpenAI client + response function
client = OpenAI()
response_fn = OpenAI_response_fn(client=client)

# 2) Create the wired RAG node pair (retrieval -> llm)
#    - It expects your input state to contain a key "user_query" by default.
rag_nodes = create_rag_query_pair(
    path="vector_db",              # ChromaDB persist directory
    collection_name="my_docs",     # existing ChromaDB collection
    response_fn=response_fn,
    llm_node_name="llm",           # name for the LLM node in the graph
)

# 3) Build and run the graph
runner = GraphRunner.build(
    node_dicts=[rag_nodes],
    start_node="retrieval",        # factory creates a node named "retrieval"
)

result = runner.execute({"user_query": "How do I implement RAG with GraphRunner?"})
print(result["state_dict"]["answer"])  # LLM answer
```

Custom prompt template (the factory will inject retrieved context automatically):

```python
from openai import OpenAI

from llm_graph.llm.response_functions import OpenAI_response_fn
from llm_graph.factories.rag import create_rag_query_pair
from llm_graph.core.graphrunner import GraphRunner

client = OpenAI()
response_fn = OpenAI_response_fn(client=client)

custom_prompt = """
You are a helpful assistant. Use the provided context to answer the question.
Respond in JSON with key 'answer'.

Context:
{retrieved_context}

The query is: {user_query}
"""

rag_nodes = create_rag_query_pair(
    path="vector_db",
    collection_name="my_docs",
    response_fn=response_fn,
    llm_node_name="llm",
    prompt_template=custom_prompt,
)

runner = GraphRunner.build(node_dicts=[rag_nodes], start_node="retrieval")
result = runner.execute({"user_query": "What is GraphRunner.build used for?"})
print(result["state_dict"]["answer"])
```

Note: the retrieval node and LLM node both default to reading the query from state key "user_query", so your input dict must include that key (or you must configure matching query_key values if you override them).

In [10]:
# Another general question
r3 = session.execute({"user_query": "What are the parse-error and tool-error retry systems in the tool factory, and how do they relate to each other?"})
Markdown(r3["state_dict"]["answer"])

The tool factory includes two retry subsystems that handle different failure modes in an LLM→tool workflow, and they are designed to chain together into a single robust loop.

Parse-error retry system (malformed JSON from the LLM)
- Problem it handles: the LLM’s tool-argument output cannot be parsed as valid JSON.
- Built by: create_retry_parse_error_pair(...)
- What it creates:
  1) A ConditionalNode that checks state["parse_error"].
     - If parse_error == True → route to a retry LLM node.
     - If parse_error == False → route to the actual tool node.
  2) A retry LLM node that includes the raw bad output + the parse error message in its prompt, and then loops back to the parse-check ConditionalNode.
- Effect: forms a loop that keeps re-asking the LLM for correctly formatted JSON until parsing succeeds (or you hit external limits).

Tool-error retry system (tool execution failure)
- Problem it handles: the tool call fails after parsing succeeded—either the tool raises an exception or it reports failure via a boolean state key like state["{output_key}_success"] == False.
- Built by: create_retry_tool_error_pair(...)
- What it creates:
  1) A ConditionalNode that checks state["{output_key}_success"].
     - On success → route to the tool_analysis_node.
     - On failure → route to a retry LLM node.
  2) A retry LLM node that includes the tool error + the attempted args in its prompt.
     - Crucially, this retry node routes next to check_parse_name (the parse-error conditional), so the new retry output is validated for JSON correctness.

How they relate
- They cover different stages:
  - Parse-error retry fixes “can’t parse the LLM output into args.”
  - Tool-error retry fixes “args parsed, but the tool call still failed.”
- They are connected deliberately:
  - The tool-error retry LLM sends its output into the parse-error check (check_parse_name), ensuring that any new arguments produced during a tool-error retry are also JSON-validated.
- Net result: a combined retry loop where tool failures trigger a new round of argument generation, and that generation is always guarded by the parse-error retry loop before the tool is attempted again.

In [11]:
# Another coding question
r4 = session.execute({"user_query": "Show me how to use the model parameter in the @tool_call decorator with Pydantic validation."})
Markdown(r4["state_dict"]["answer"])

```python
from pydantic import BaseModel, Field

from llm_graph.tools.tool_call import tool_call

# 1) Define a Pydantic model that describes/validates the tool arguments
class SearchArgs(BaseModel):
    query: str = Field(..., min_length=1)
    n_results: int = Field(3, ge=1, le=10)

# 2) Decorate your function with @tool_call and pass model=...
#    - The runtime will read kwargs from state["search_params"]
#    - It will validate them against SearchArgs before calling the function
#    - The return value will be written to state["search_results"] in the delta
@tool_call(input_key="search_params", output_key="search_results", model=SearchArgs)
def search_database(query: str, n_results: int = 3):
    # ... your real retrieval logic here ...
    return {"hits": [f"Result {i} for '{query}'" for i in range(1, n_results + 1)]}

# Example: calling the decorated tool with a state dict
state = {
    "search_params": {
        "query": "GraphRunner.build",
        "n_results": 3,
    }
}

# The decorated function is called with the whole state dict
delta = search_database(state)
print(delta["search_results"])  # {'hits': [...]} 

# Example: invalid args fail validation (returns a failure result instead of raising)
bad_state = {"search_params": {"query": "", "n_results": 999}}
failed_delta = search_database(bad_state)
print(failed_delta)  # will contain a failure result under output_key (implementation-defined)

# (Optional) tool metadata attached by the decorator (used by factories)
print(search_database.tool_meta["input_key"])      # 'search_params'
print(search_database.tool_meta["output_key"])     # 'search_results'
print(search_database.tool_meta["schema_model"])   # <class '__main__.SearchArgs'>
```


## Inspect the trace

In [12]:
# Show the execution path for the last query
graphrunner.print_trace()

Step 1
  Node: branch_llm
  Node type: FunctionalNode
  Input: {'message_history': [{'role': 'user', 'content': "\nYou are making a decision on the type of query given.\nThis will either be a coding query (asking for code examples, implementation details, usage snippets)\nor a general query (asking for conceptual explanations, descriptions, or comparisons).\n\nRespond ONLY with valid JSON with key 'query_type' set to either 'coding' or 'general'.\n\nThe query is: How does GraphRunner.build() work and when would I use it instead of the regular constructor?\n"}, {'role': 'assistant', 'content': '{"query_type":"general"}'}, {'role': 'user', 'content': "\nUse the following retrieved context to answer the question.\nContext :\nComponent documentation: GraphRunner\n\n### Component: GraphRunner\n\nPurpose:\nGraphRunner manages execution of workflow graphs by controlling node traversal\nand maintaining runtime state. It also tracks token usage and enforces optional node visit limits. It does n

In [13]:
# Token usage across the whole session
print(f"In tokens : {session.in_tokens}")
print(f"Out tokens: {session.out_tokens}")

In tokens : 19781
Out tokens: 2017
